# Pocket Analysis with TopoMT

Protein pockets are regions of the molecular surface that can host small-molecule ligands.
The **topomt** addon integrates TopoMT's Delaunay-based pocket detection with MolSysViewer
to let you:

- compute a full topography (all pockets + volume/area metrics),
- visualise all pockets as semi-transparent blobs,
- zoom into a specific pocket by feature ID.

The pharmacophore tutorial (`tutorial_pharmacophoremt_overlay.ipynb`) continues from
this point and builds a pharmacophore model inside the pocket identified here.

**Requirements:** `topomt`, `molsysmt`, `molsysviewer`, `molsysviewer-topomt`.

In [1]:
import molsysmt as msm
import topomt
import molsysviewer as msv
from molsysviewer_topomt import (
    get_addon,
    lifecycle,
    on_enable,
    build_view_with_topography,
    attach_topography,
)
from molsysviewer_topomt.runtime import ensure_runtime

In [2]:
msv.addons.register(get_addon(), lifecycle=lifecycle)

AddonSpec(name='topomt', package='molsysviewer-topomt', version='0.1.0', description='TopoMT workspace for pocket and topography analysis in MolSysViewer.', workspaces=(AddonWorkspaceSpec(id='topomt', title='TopoMT', entry_panel='topography', description='Topography-focused workspace for pocket analysis workflows.', order=20, meta={}),), panels=(AddonPanelSpec(id='topography', title='Topography', entry='molsysviewer_topomt.panels.topography_panel', widget_class='molsysviewer_topomt.panels.topography.TopoMTTopographyPanel', description='Summary panel with pocket render controls.', order=10, target='panel_mode', meta={}), AddonPanelSpec(id='pockets', title='Pockets', entry='molsysviewer_topomt.panels.pockets_panel', widget_class='molsysviewer_topomt.panels.pockets.TopoMTPocketsPanel', description='Per-pocket list with individual show/hide controls.', order=20, target='panel_mode', meta={})), context_actions=(AddonContextActionSpec(id='focus-topography-feature', title='Focus Topography Fe

## Load the protein and compute the topography

We use CDK2 (PDB 1FIN), a serine/threonine kinase with a well-characterised
ATP-binding pocket.  `topomt.get_topography()` uses the Pocketeer algorithm by
default; only the first structure (frame 0) is used for pocket detection.

In [3]:
ms = msm.convert(
    "pdb_id:1fin",
    to_form="molsysmt.MolSys",
    selection='molecule_type == "protein"',
)

topography = topomt.get_topography(ms, method="pocketeer", structure_indices=0)
print(f"Pockets detected: {len(topography.features)}")

Pockets detected: 31


## Inspect pocket metrics

Each feature in the topography carries volume, area, and hydrophobicity estimates.
Sorting by volume helps identify the dominant binding site.

In [4]:
for feat in sorted(topography.features.values(), key=lambda f: f.volume, reverse=True)[:5]:
    print(f"  id={feat.id}  volume={feat.volume:.1f} Å³")

  id=POC-2  volume=2.0 nanometer ** 3 Å³
  id=POC-1  volume=1.8 nanometer ** 3 Å³
  id=POC-5  volume=1.8 nanometer ** 3 Å³
  id=POC-6  volume=1.4 nanometer ** 3 Å³
  id=POC-4  volume=1.4 nanometer ** 3 Å³


## Build a view with all pockets overlaid

`build_view_with_topography()` is a one-call convenience that creates the viewer,
loads the protein, registers and enables the addon, and renders all pocket blobs.

In [5]:
view = build_view_with_topography(
    ms,
    topography,
    selection='molecule_type == "protein"',
)
view.show()

## Focus on a specific pocket

To isolate the largest pocket (the ATP-binding site), clear the full-topography render
and redraw only the feature of interest via the **Pockets** panel.

In [6]:
pockets_panel = view.addons.resolve_panel_widget("topomt", "pockets")

# Clear all pocket shapes first
pockets_panel.handle_action(view, "clear_pockets", {})

# Render only the largest pocket (feature id from the sorted list above)
largest_pocket_id = sorted(topography.features.values(), key=lambda f: f.volume, reverse=True)[0].id
pockets_panel.handle_action(view, "show_pocket", {"feature_id": largest_pocket_id})

print(f"Showing pocket id={largest_pocket_id}")

Showing pocket id=POC-2


## Export a static snapshot

In [7]:
view.export.html("1fin_pocket.html", title="CDK2 — ATP-binding pocket")

## Next steps

The identified pocket is the natural starting point for pharmacophore modelling.
Continue to `tutorial_pharmacophoremt_overlay.ipynb`, which builds a structure-based
pharmacophore inside this pocket and overlays the interaction-site glyphs in the
same view.